In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()
session.sql("USE DATABASE NHANES_DB").collect()
session.sql("USE SCHEMA RAW").collect()

df = session.table("NHANES_FINAL").to_pandas()

# Clean
df_clean = df.dropna(subset=['SYSTOLIC_BP', 'DIASTOLIC_BP'])
diet_cols = ['SODIUM_MG','FIBER_G','SUGAR_G','CALORIES','PROTEIN_G','FAT_G']
df_clean[diet_cols] = df_clean[diet_cols].fillna(df_clean[diet_cols].median())
df_clean['BMI'] = df_clean['BMI'].fillna(df_clean['BMI'].median())

print("Ready! Shape:", df_clean.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd

# Features and target
features = ['SODIUM_MG', 'CALORIES', 'FIBER_G', 'BMI',
            'AGE', 'PROTEIN_G', 'FAT_G',
            'GENDER', 'ETHNICITY',
            'WEIGHT_KG', 'HEIGHT_CM']
target = 'HYPERTENSION_FLAG'

# Prepare data
ml_df = df_clean.dropna(subset=[target] + features)
X = ml_df[features]
y = ml_df[target]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate
rf_pred = rf_model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("\nClassification Report:\n", classification_report(y_test, rf_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, rf_pred))

# Feature importance
pd.Series(rf_model.feature_importances_, index=features).sort_values().plot(kind='barh')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()